# NewsBot Intelligence System
## ITAI 2373 Midterm Project

**Team Members:** Jason Trimble - solo project  
**Date:** June 28, 2026  
**Repository URL:** TODO: Add your GitHub repository URL after pushing the repo

This notebook builds a complete NewsBot Intelligence System using the HuffPost News Category Dataset. Because the source data contains headlines and short descriptions rather than full articles, the system treats `headline + short_description` as the article text used for preprocessing, analysis, and classification.


## Setup

This notebook is designed to run in a local Jupyter environment or in Google Colab. The repository includes a prepared balanced sample in `data/newsbot_dataset_sample.csv` so the analysis stays within typical free-tier hardware limits while still meeting the assignment requirements.

If you are starting from a clean environment, keep the next cell enabled so it can install any missing packages automatically.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "nltk": "nltk",
    "spacy": "spacy",
    "wordcloud": "wordcloud",
    "plotly": "plotly",
}

missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")


In [ ]:
import html
import json
import random
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from IPython.display import display
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from scipy.sparse import csr_matrix, hstack
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")

CWD = Path.cwd()
PROJECT_DIR_CANDIDATES = [
    CWD,
    CWD / "ITAI2373-NewsBot-Midterm",
]
PROJECT_DIR = next(
    (
        candidate
        for candidate in PROJECT_DIR_CANDIDATES
        if (candidate / "Midterm_NewsBot_Intelligence_System_completed.ipynb").exists()
        or (candidate / "data").exists()
    ),
    CWD,
)
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)

RESOURCE_LOOKUPS = {
    "punkt": ["tokenizers/punkt", "tokenizers/punkt.zip"],
    "punkt_tab": ["tokenizers/punkt_tab", "tokenizers/punkt_tab.zip"],
    "stopwords": ["corpora/stopwords", "corpora/stopwords.zip"],
    "wordnet": ["corpora/wordnet", "corpora/wordnet.zip", "corpora/wordnet.zip/wordnet"],
    "omw-1.4": ["corpora/omw-1.4", "corpora/omw-1.4.zip"],
    "vader_lexicon": [
        "sentiment/vader_lexicon",
        "sentiment/vader_lexicon.zip",
        "sentiment/vader_lexicon.zip/vader_lexicon/vader_lexicon.txt",
    ],
    "averaged_perceptron_tagger": [
        "taggers/averaged_perceptron_tagger",
        "taggers/averaged_perceptron_tagger.zip",
    ],
    "averaged_perceptron_tagger_eng": [
        "taggers/averaged_perceptron_tagger_eng",
        "taggers/averaged_perceptron_tagger_eng.zip",
    ],
}

for resource_name, candidate_paths in RESOURCE_LOOKUPS.items():
    resource_available = False
    for candidate_path in candidate_paths:
        try:
            nltk.data.find(candidate_path)
            resource_available = True
            break
        except LookupError:
            continue

    if not resource_available:
        try:
            nltk.download(resource_name, quiet=True)
        except Exception as exc:
            print(f"Could not download {resource_name}: {exc}")

SPACY_MODEL = "en_core_web_sm"
try:
    nlp = spacy.load(SPACY_MODEL)
except OSError:
    print(f"Downloading spaCy model: {SPACY_MODEL}")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", SPACY_MODEL])
    nlp = spacy.load(SPACY_MODEL)

print("Environment ready.")


## Module 1: Real-World NLP Application Context

**What was done:** Framed the NewsBot system as a media intelligence workflow and loaded a balanced six-category dataset sample.  
**Why it matters:** A business use case gives the technical pipeline a clear objective instead of treating each NLP method as an isolated exercise.  
**Key finding:** The selected HuffPost categories give enough variety for routing, sentiment tracking, linguistic comparison, and entity analysis.  
**Business interpretation:** Editors, researchers, marketers, and analysts could use this system for first-pass content routing, trend monitoring, and competitor coverage tracking.


In [ ]:
SELECTED_CATEGORIES = [
    "POLITICS",
    "ENTERTAINMENT",
    "BUSINESS",
    "SPORTS",
    "TECH",
    "WELLNESS",
]
ROWS_PER_CATEGORY = 300
MIN_WORDS = 15
FORCE_REBUILD_SAMPLE = False

SAMPLE_CANDIDATES = [
    PROJECT_DIR / "data" / "newsbot_dataset_sample.csv",
    PROJECT_DIR / "newsbot_dataset_sample.csv",
]

RAW_DATA_CANDIDATES = [
    PROJECT_DIR / "data" / "News_Category_Dataset_v3.json",
    PROJECT_DIR / "News_Category_Dataset_v3.json",
    PROJECT_DIR.parent / "archive" / "News_Category_Dataset_v3.json",
    CWD / "archive" / "News_Category_Dataset_v3.json",
    Path("/content/News_Category_Dataset_v3.json"),
    Path("/mnt/data/News_Category_Dataset_v3.json"),
]

def first_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None

sample_path = first_existing_path(SAMPLE_CANDIDATES)
raw_path = first_existing_path(RAW_DATA_CANDIDATES)

if sample_path is not None and not FORCE_REBUILD_SAMPLE:
    df = pd.read_csv(sample_path)
    dataset_source = f"Prepared balanced sample: {display_path(sample_path)}"
else:
    if raw_path is None:
        raise FileNotFoundError(
            "Could not find either data/newsbot_dataset_sample.csv or News_Category_Dataset_v3.json."
        )

    raw_df = pd.read_json(raw_path, lines=True)
    df = raw_df[raw_df["category"].isin(SELECTED_CATEGORIES)].copy()
    df["headline"] = df["headline"].fillna("").astype(str)
    df["short_description"] = df["short_description"].fillna("").astype(str)
    df["authors"] = df["authors"].fillna("Unknown").replace("", "Unknown")
    df["link"] = df["link"].fillna("").astype(str)
    df["title"] = df["headline"].str.strip()
    df["content"] = df["short_description"].str.strip()
    df["full_text"] = (df["title"] + " " + df["content"]).str.replace(r"\s+", " ", regex=True).str.strip()
    df["source"] = "HuffPost"
    df = df[(df["title"].str.len() > 0) & (df["content"].str.len() > 0)].copy()
    df = df[df["full_text"].str.split().str.len() >= MIN_WORDS].copy()

    balanced_parts = []
    for category in SELECTED_CATEGORIES:
        category_df = df[df["category"] == category].copy()
        sample_size = min(ROWS_PER_CATEGORY, len(category_df))
        balanced_parts.append(category_df.sample(n=sample_size, random_state=RANDOM_STATE))

    df = pd.concat(balanced_parts, ignore_index=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    df["article_id"] = range(1, len(df) + 1)
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    df = df[
        [
            "article_id",
            "title",
            "content",
            "full_text",
            "category",
            "authors",
            "source",
            "date",
            "link",
        ]
    ]
    output_sample_path = DATA_DIR / "newsbot_dataset_sample.csv"
    df.to_csv(output_sample_path, index=False)
    dataset_source = f"Raw HuffPost dataset rebuilt and saved to: {display_path(output_sample_path)}"

if "article_id" not in df.columns:
    df["article_id"] = range(1, len(df) + 1)

for column in ["title", "content", "full_text", "category", "authors", "source", "date", "link"]:
    if column not in df.columns:
        df[column] = ""

df["title"] = df["title"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str)
df["full_text"] = df["full_text"].fillna(df["title"] + " " + df["content"]).astype(str)
df["category"] = df["category"].astype(str)
df["word_count"] = df["full_text"].str.split().str.len()
df["char_count"] = df["full_text"].str.len()

print(dataset_source)
print(f"Dataset shape: {df.shape}")
print("Category counts:")
print(df["category"].value_counts().reindex(SELECTED_CATEGORIES))
display(df.head(3))


In [ ]:
print("Dataset overview")
print("-" * 60)
print(f"Articles: {len(df)}")
print(f"Categories: {df['category'].nunique()}")
print(f"Average words per article: {df['word_count'].mean():.2f}")
print(f"Average characters per article: {df['char_count'].mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

category_counts = df["category"].value_counts().reindex(SELECTED_CATEGORIES)
sns.barplot(
    x=category_counts.index,
    y=category_counts.values,
    ax=axes[0],
    hue=category_counts.index,
    legend=False,
)
axes[0].set_title("Category Distribution")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Article Count")
axes[0].tick_params(axis="x", rotation=30)

sns.histplot(df["word_count"], bins=30, kde=True, ax=axes[1], color="#1f77b4")
axes[1].set_title("Text Length Distribution")
axes[1].set_xlabel("Words in headline + short description")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "category_distribution.png", dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "text_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    "Business note: The balanced sample keeps the classifier fair across categories and makes downstream comparisons easier to interpret."
)


## Module 2: Text Preprocessing Pipeline

**What was done:** Built a reusable cleaning and normalization pipeline for article text.  
**Why it matters:** Raw news text contains punctuation, URLs, markup, and filler words that can dilute feature quality.  
**Key finding:** Preprocessing reduces noise and keeps the signal needed for TF-IDF, classification, and linguistic pattern analysis.  
**Business interpretation:** Cleaner text means more stable routing decisions and more reliable downstream insights.


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def preprocess_text(text, remove_stopwords=True, lemmatize=True):
    cleaned = clean_text(text)
    if not cleaned:
        return ""

    tokens = word_tokenize(cleaned)

    if remove_stopwords:
        tokens = [token for token in tokens if token not in stop_words]

    if lemmatize:
        tokens = [lemmatizer.lemmatize(token) for token in tokens]

    tokens = [token for token in tokens if len(token) > 2]
    return " ".join(tokens)

preview_text = df.iloc[0]["full_text"]
print("Original sample:")
print(preview_text)
print("\nProcessed sample:")
print(preprocess_text(preview_text))


In [ ]:
df["clean_full_text"] = df["full_text"].apply(clean_text)
df["processed_text"] = df["full_text"].apply(preprocess_text)
df["processed_word_count"] = df["processed_text"].str.split().str.len()
df["processed_char_count"] = df["processed_text"].str.len()

print("Preprocessing summary")
print("-" * 60)
print(f"Average words before preprocessing: {df['word_count'].mean():.2f}")
print(f"Average words after preprocessing: {df['processed_word_count'].mean():.2f}")
print(f"Average characters before preprocessing: {df['char_count'].mean():.2f}")
print(f"Average characters after preprocessing: {df['processed_char_count'].mean():.2f}")

all_tokens = " ".join(df["processed_text"]).split()
common_terms = pd.DataFrame(Counter(all_tokens).most_common(20), columns=["term", "count"])
display(common_terms)

plt.figure(figsize=(12, 6))
sns.barplot(data=common_terms, x="count", y="term", color="#4c72b0")
plt.title("Top 20 Most Common Processed Terms")
plt.xlabel("Frequency")
plt.ylabel("Term")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "preprocessing_common_terms.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    "Key finding: the preprocessing pipeline compresses noisy surface text into a smaller, more informative vocabulary for modeling."
)


## Module 3: TF-IDF Feature Extraction and Analysis

**What was done:** Converted processed text into a TF-IDF matrix with unigram and bigram features.  
**Why it matters:** TF-IDF highlights terms that are important to a category instead of merely frequent across the entire dataset.  
**Key finding:** Each category develops a distinct vocabulary footprint that supports both interpretation and classification.  
**Business interpretation:** These terms show what language patterns distinguish topic clusters, which is useful for newsroom routing and trend monitoring.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.8,
    stop_words="english",
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["processed_text"])
feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix created")
print("-" * 60)
print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(feature_names)}")
print(f"Non-zero entries: {tfidf_matrix.nnz}")


In [ ]:
top_terms_by_category = {}
top_term_records = []

for category in SELECTED_CATEGORIES:
    category_mask = df["category"].values == category
    category_mean_scores = np.asarray(tfidf_matrix[category_mask].mean(axis=0)).ravel()
    top_indices = category_mean_scores.argsort()[-10:][::-1]
    category_terms = []

    for rank, feature_index in enumerate(top_indices, start=1):
        term = feature_names[feature_index]
        score = float(category_mean_scores[feature_index])
        category_terms.append({"term": term, "score": score})
        top_term_records.append(
            {
                "category": category,
                "rank": rank,
                "term": term,
                "score": score,
            }
        )

    top_terms_by_category[category] = category_terms

top_terms_df = pd.DataFrame(top_term_records)
display(top_terms_df.head(18))

fig, axes = plt.subplots(len(SELECTED_CATEGORIES), 1, figsize=(12, 18), sharex=False)
for axis, category in zip(axes, SELECTED_CATEGORIES):
    plot_df = top_terms_df[top_terms_df["category"] == category].head(8).sort_values("score")
    axis.barh(plot_df["term"], plot_df["score"], color="#2a9d8f")
    axis.set_title(f"Top TF-IDF Terms: {category}")
    axis.set_xlabel("Average TF-IDF score")
    axis.set_ylabel("")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tfidf_top_terms.png", dpi=150, bbox_inches="tight")
plt.show()

representative_terms = {
    category: ", ".join(item["term"] for item in top_terms_by_category[category][:5])
    for category in SELECTED_CATEGORIES
}
print("Representative terms by category:")
for category, terms in representative_terms.items():
    print(f"- {category}: {terms}")

print("Interpretation: TF-IDF shows that the categories are linguistically distinct enough for a multi-class classifier to learn meaningful boundaries.")


## Module 4: Part-of-Speech Pattern Analysis

**What was done:** Tagged article text and measured category-level proportions for nouns, proper nouns, verbs, adjectives, and adverbs.  
**Why it matters:** POS patterns reveal stylistic differences that simple keyword counts miss.  
**Key finding:** News categories differ not only in vocabulary but also in how they structure information.  
**Business interpretation:** Style differences can help explain why some categories are easier to classify and how journalists frame different topics.


In [ ]:
POS_GROUPS = {
    "nouns": {"NN", "NNS"},
    "proper_nouns": {"NNP", "NNPS"},
    "verbs": {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"},
    "adjectives": {"JJ", "JJR", "JJS"},
    "adverbs": {"RB", "RBR", "RBS"},
}

def analyze_pos_patterns(text):
    tokens = [token for token in word_tokenize(str(text)) if re.search(r"[A-Za-z]", token)]
    if not tokens:
        return {group: 0.0 for group in POS_GROUPS}

    tagged_tokens = pos_tag(tokens)
    total_tokens = len(tagged_tokens)
    counts = {group: 0 for group in POS_GROUPS}

    for _, tag in tagged_tokens:
        for group_name, group_tags in POS_GROUPS.items():
            if tag in group_tags:
                counts[group_name] += 1

    return {group: counts[group] / total_tokens for group in POS_GROUPS}

pos_records = []
for row in df.itertuples(index=False):
    pos_result = analyze_pos_patterns(row.full_text)
    pos_result["article_id"] = row.article_id
    pos_result["category"] = row.category
    pos_records.append(pos_result)

pos_article_df = pd.DataFrame(pos_records)
print("POS analysis complete.")
display(pos_article_df.head())


In [ ]:
pos_category_summary = pos_article_df.groupby("category")[
    ["nouns", "proper_nouns", "verbs", "adjectives", "adverbs"]
].mean().reindex(SELECTED_CATEGORIES)

display(pos_category_summary.round(4))

pos_plot_df = pos_category_summary.reset_index().melt(
    id_vars="category",
    var_name="pos_group",
    value_name="proportion",
)

plt.figure(figsize=(12, 6))
sns.barplot(data=pos_plot_df, x="category", y="proportion", hue="pos_group")
plt.title("POS Pattern Comparison by Category")
plt.xlabel("Category")
plt.ylabel("Average token proportion")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pos_patterns_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

most_proper_nouns = pos_category_summary["proper_nouns"].idxmax()
most_adjectives = pos_category_summary["adjectives"].idxmax()
print(f"Category with the highest proper noun usage: {most_proper_nouns}")
print(f"Category with the highest adjective usage: {most_adjectives}")
print("Interpretation: proper nouns usually signal people, teams, organizations, and places, while adjective-heavy writing often reflects descriptive or evaluative coverage.")


## Module 5: Syntax Parsing and Semantic Analysis

**What was done:** Used spaCy dependency parsing on a stratified sample to measure sentence structure, noun phrases, subjects, objects, and dependency patterns.  
**Why it matters:** Syntax captures how information is organized, not just which words appear.  
**Key finding:** Category differences appear in sentence complexity and how often articles foreground actors versus events.  
**Business interpretation:** These patterns help explain why some categories sound more report-like, descriptive, or event-driven than others.


In [ ]:
SYNTAX_SAMPLE_PER_CATEGORY = 30

syntax_sample_parts = []
for category in SELECTED_CATEGORIES:
    category_df = df[df["category"] == category].copy()
    syntax_sample_parts.append(
        category_df.sample(n=min(SYNTAX_SAMPLE_PER_CATEGORY, len(category_df)), random_state=RANDOM_STATE)
    )

syntax_sample_df = pd.concat(syntax_sample_parts, ignore_index=True)

def extract_syntactic_features(doc):
    sentence_lengths = [
        len([token for token in sent if not token.is_space and not token.is_punct])
        for sent in doc.sents
    ]
    return {
        "num_sentences": len(sentence_lengths),
        "avg_sentence_length": float(np.mean(sentence_lengths)) if sentence_lengths else 0.0,
        "noun_phrase_count": sum(1 for _ in doc.noun_chunks),
        "subject_count": sum(1 for token in doc if token.dep_ in {"nsubj", "nsubjpass", "csubj"}),
        "object_count": sum(1 for token in doc if token.dep_ in {"dobj", "iobj", "pobj", "obj"}),
    }

syntax_records = []
dependency_records = []

syntax_docs = nlp.pipe(syntax_sample_df["full_text"].tolist(), batch_size=32)
for row, doc in zip(syntax_sample_df.itertuples(index=False), syntax_docs):
    features = extract_syntactic_features(doc)
    features["article_id"] = row.article_id
    features["category"] = row.category
    syntax_records.append(features)

    for token in doc:
        if not token.is_space and not token.is_punct:
            dependency_records.append({"category": row.category, "dependency": token.dep_})

syntax_df = pd.DataFrame(syntax_records)
dependency_df = pd.DataFrame(dependency_records)

print(
    f"Syntactic analysis completed on {len(syntax_sample_df)} articles "
    f"({SYNTAX_SAMPLE_PER_CATEGORY} per category where available)."
)
display(syntax_df.head())


In [ ]:
syntax_category_summary = syntax_df.groupby("category")[
    ["num_sentences", "avg_sentence_length", "noun_phrase_count", "subject_count", "object_count"]
].mean().reindex(SELECTED_CATEGORIES)
display(syntax_category_summary.round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
syntax_metrics = [
    ("avg_sentence_length", "Average Sentence Length"),
    ("noun_phrase_count", "Average Noun Phrases"),
    ("subject_count", "Average Subjects"),
    ("object_count", "Average Objects"),
]

for axis, (column_name, title) in zip(axes.flatten(), syntax_metrics):
    plot_values = syntax_category_summary[column_name]
    axis.bar(plot_values.index, plot_values.values, color="#f4a261")
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "syntax_features_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

top_dependencies = (
    dependency_df.groupby(["category", "dependency"])
    .size()
    .reset_index(name="count")
    .sort_values(["category", "count"], ascending=[True, False])
    .groupby("category")
    .head(5)
)
display(top_dependencies)

longest_sentences = syntax_category_summary["avg_sentence_length"].idxmax()
richest_noun_phrases = syntax_category_summary["noun_phrase_count"].idxmax()
print(f"Category with the longest average sentences: {longest_sentences}")
print(f"Category with the most noun phrases on average: {richest_noun_phrases}")
print("Interpretation: dependency parsing adds structure-aware evidence that complements TF-IDF and POS features.")


## Module 6: Sentiment and Emotion Analysis

**What was done:** Applied VADER sentiment analysis to score each article and assign positive, neutral, or negative labels.  
**Why it matters:** Sentiment adds an emotional lens that keyword-based methods do not capture.  
**Key finding:** Different categories carry noticeably different emotional tone profiles.  
**Business interpretation:** Editors and analysts can use sentiment patterns to monitor tone shifts, reputation risk, and audience-facing content balance.


In [ ]:
sia = SentimentIntensityAnalyzer()

def analyze_sentiment(text):
    if not text or pd.isna(text):
        return {
            "compound": 0.0,
            "pos": 0.0,
            "neu": 1.0,
            "neg": 0.0,
            "sentiment_label": "neutral",
        }

    scores = sia.polarity_scores(str(text))
    if scores["compound"] >= 0.05:
        label = "positive"
    elif scores["compound"] <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    scores["sentiment_label"] = label
    return scores

sentiment_records = []
for row in df.itertuples(index=False):
    scores = analyze_sentiment(row.full_text)
    sentiment_records.append(
        {
            "article_id": row.article_id,
            "category": row.category,
            "compound": scores["compound"],
            "pos": scores["pos"],
            "neu": scores["neu"],
            "neg": scores["neg"],
            "sentiment_label": scores["sentiment_label"],
        }
    )

sentiment_df = pd.DataFrame(sentiment_records)
display(sentiment_df.head())


In [ ]:
sentiment_summary = (
    sentiment_df.groupby("category")
    .agg(
        mean_sentiment=("compound", "mean"),
        median_sentiment=("compound", "median"),
        positive_share=("sentiment_label", lambda values: np.mean(values == "positive")),
        negative_share=("sentiment_label", lambda values: np.mean(values == "negative")),
    )
    .reindex(SELECTED_CATEGORIES)
)
display(sentiment_summary.round(3))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=sentiment_summary.reset_index(),
    x="category",
    y="mean_sentiment",
    hue="category",
    legend=False,
)
plt.title("Average Sentiment by Category")
plt.xlabel("Category")
plt.ylabel("Average compound sentiment")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sentiment_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

sentiment_label_counts = (
    sentiment_df.groupby(["category", "sentiment_label"])
    .size()
    .reset_index(name="count")
)

plt.figure(figsize=(11, 6))
sns.barplot(data=sentiment_label_counts, x="category", y="count", hue="sentiment_label")
plt.title("Sentiment Label Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Article count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sentiment_label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

most_positive_category = sentiment_summary["mean_sentiment"].idxmax()
most_negative_category = sentiment_summary["mean_sentiment"].idxmin()
print(f"Most positive category: {most_positive_category}")
print(f"Most negative category: {most_negative_category}")
print("Business note: category-level tone can help monitor coverage balance, brand risk, and editorial framing.")


## Module 7: Multi-Class Text Classification System

**What was done:** Combined TF-IDF, sentiment, and length features, then compared three classifiers.  
**Why it matters:** Model comparison shows whether performance comes from the features, the algorithm, or both.  
**Key finding:** The best-performing model creates a strong baseline for automated article routing.  
**Business interpretation:** A reliable classifier reduces manual triage time and makes large-scale media monitoring practical.


In [ ]:
numeric_features = np.column_stack(
    [
        sentiment_df["compound"].values,
        sentiment_df["pos"].values,
        sentiment_df["neu"].values,
        sentiment_df["neg"].values,
        df["char_count"].values,
        df["word_count"].values,
        df["title"].str.len().values,
    ]
)

numeric_scaler = StandardScaler()
numeric_features_scaled = numeric_scaler.fit_transform(numeric_features)

X_tfidf = tfidf_matrix
X_combined = hstack([X_tfidf, csr_matrix(numeric_features_scaled)], format="csr")
y = df["category"].values
article_indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    article_indices,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_tfidf = X_tfidf[train_idx]
X_test_tfidf = X_tfidf[test_idx]
X_train_combined = X_combined[train_idx]
X_test_combined = X_combined[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("Feature matrices prepared.")
print(f"TF-IDF train shape: {X_train_tfidf.shape}")
print(f"Combined feature train shape: {X_train_combined.shape}")


In [ ]:
model_specs = {
    "Multinomial Naive Bayes": {
        "model": MultinomialNB(),
        "X_train": X_train_tfidf,
        "X_test": X_test_tfidf,
    },
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE,
            solver="saga",
        ),
        "X_train": X_train_combined,
        "X_test": X_test_combined,
    },
    "Linear SVM": {
        "model": LinearSVC(random_state=RANDOM_STATE),
        "X_train": X_train_combined,
        "X_test": X_test_combined,
    },
}

model_results = []
trained_models = {}
prediction_store = {}

for model_name, spec in model_specs.items():
    model = clone(spec["model"])
    model.fit(spec["X_train"], y_train)
    predictions = model.predict(spec["X_test"])

    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(y_test, predictions, average="macro")
    weighted_f1 = f1_score(y_test, predictions, average="weighted")
    cv_scores = cross_val_score(
        clone(spec["model"]),
        spec["X_train"],
        y_train,
        cv=3,
        scoring="f1_macro",
        n_jobs=-1,
    )

    model_results.append(
        {
            "Model": model_name,
            "Accuracy": accuracy,
            "Macro F1": macro_f1,
            "Weighted F1": weighted_f1,
            "CV Macro F1 Mean": float(cv_scores.mean()),
            "CV Macro F1 Std": float(cv_scores.std()),
        }
    )

    trained_models[model_name] = model
    prediction_store[model_name] = predictions

model_results_df = pd.DataFrame(model_results).sort_values(
    ["Macro F1", "Accuracy"], ascending=False
)
model_results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

display(model_results_df.round(4))

best_model_name = model_results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = prediction_store[best_model_name]
print(f"Best model by Macro F1: {best_model_name}")


In [ ]:
classification_report_dict = classification_report(y_test, best_predictions, output_dict=True)
classification_report_df = pd.DataFrame(classification_report_dict).T
display(classification_report_df.round(3))

with open(OUTPUT_DIR / "classification_report.json", "w", encoding="utf-8") as handle:
    json.dump(classification_report_dict, handle, indent=2)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    normalize="true",
    xticks_rotation=30,
    cmap="Blues",
    ax=ax,
)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

deployment_candidates = []
for model_name in model_results_df["Model"]:
    if hasattr(trained_models[model_name], "predict_proba"):
        deployment_candidates.append(model_name)

deployment_model_name = (
    model_results_df[model_results_df["Model"].isin(deployment_candidates)]
    .sort_values(["Macro F1", "Accuracy"], ascending=False)
    .iloc[0]["Model"]
)
deployment_model = trained_models[deployment_model_name]
deployment_macro_f1 = model_results_df.set_index("Model").loc[deployment_model_name, "Macro F1"]
best_macro_f1 = model_results_df.iloc[0]["Macro F1"]
deployment_feature_mode = (
    "tfidf" if deployment_model_name == "Multinomial Naive Bayes" else "combined"
)

print(f"Deployment model for the final NewsBot system: {deployment_model_name}")
print(
    f"Reason: it exposes class probabilities, and its Macro F1 is within "
    f"{best_macro_f1 - deployment_macro_f1:.4f} of the top-performing model."
)
print(f"Deployment feature mode: {deployment_feature_mode}")


## Module 8: Named Entity Recognition and Analysis

**What was done:** Used spaCy NER to extract people, organizations, locations, dates, money values, nationalities, and events.  
**Why it matters:** Entities convert unstructured news text into searchable facts.  
**Key finding:** Entity patterns differ strongly by category and add context that plain topic labels cannot provide.  
**Business interpretation:** Entity extraction enables monitoring of who is being discussed, where events happen, and which institutions dominate coverage.


In [ ]:
TARGET_ENTITY_LABELS = {"PERSON", "ORG", "GPE", "DATE", "MONEY", "NORP", "EVENT"}
NER_SAMPLE_PER_CATEGORY = 150

ner_sample_parts = []
for category in SELECTED_CATEGORIES:
    category_df = df[df["category"] == category].copy()
    ner_sample_parts.append(
        category_df.sample(n=min(NER_SAMPLE_PER_CATEGORY, len(category_df)), random_state=RANDOM_STATE)
    )

ner_sample_df = pd.concat(ner_sample_parts, ignore_index=True)

def extract_entities_from_text(text, allowed_labels=None):
    if not text or pd.isna(text):
        return []

    doc = nlp(str(text))
    entities = []
    for ent in doc.ents:
        if allowed_labels and ent.label_ not in allowed_labels:
            continue
        entities.append(
            {
                "text": ent.text,
                "label": ent.label_,
                "description": spacy.explain(ent.label_),
            }
        )
    return entities

entity_records = []
article_entity_records = []

ner_docs = nlp.pipe(ner_sample_df["full_text"].tolist(), batch_size=32)
for row, doc in zip(ner_sample_df.itertuples(index=False), ner_docs):
    article_entities = []
    for ent in doc.ents:
        if ent.label_ not in TARGET_ENTITY_LABELS:
            continue
        entity_record = {
            "article_id": row.article_id,
            "category": row.category,
            "text": ent.text,
            "label": ent.label_,
            "description": spacy.explain(ent.label_),
        }
        entity_records.append(entity_record)
        article_entities.append(entity_record)

    article_entity_records.append(
        {
            "article_id": row.article_id,
            "category": row.category,
            "entity_count": len(article_entities),
        }
    )

entities_df = pd.DataFrame(entity_records)
article_entities_df = pd.DataFrame(article_entity_records)

print(
    f"NER processed {len(ner_sample_df)} articles "
    f"({NER_SAMPLE_PER_CATEGORY} per category where available)."
)
print(f"Total extracted entities: {len(entities_df)}")
display(entities_df.head(10))


In [ ]:
if entities_df.empty:
    raise ValueError("No entities were extracted. Check the spaCy model installation.")

entity_type_counts = entities_df["label"].value_counts().reset_index()
entity_type_counts.columns = ["label", "count"]

plt.figure(figsize=(10, 5))
sns.barplot(data=entity_type_counts, x="label", y="count", hue="label", legend=False)
plt.title("Entity Type Distribution")
plt.xlabel("Entity label")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "entity_type_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

entity_heatmap = (
    entities_df.groupby(["category", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=SELECTED_CATEGORIES, fill_value=0)
)

plt.figure(figsize=(10, 6))
sns.heatmap(entity_heatmap, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Entity Types by Category")
plt.xlabel("Entity label")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "entity_type_by_category_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

top_entities = (
    entities_df.groupby(["label", "text"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(15)
)
display(top_entities)

print("Interpretation:")
print("- POLITICS and BUSINESS should surface many ORG, PERSON, and GPE mentions.")
print("- SPORTS often includes PERSON, DATE, and EVENT mentions tied to matches and athletes.")
print("- Entity extraction turns the classifier into a richer intelligence tool instead of a simple labeler.")


## Comprehensive Analysis and Business Insights

**What was done:** Combined results from preprocessing, TF-IDF, syntax, sentiment, classification, and NER into a single summary layer.  
**Why it matters:** The assignment asks for an integrated system, not disconnected notebook sections.  
**Key finding:** The strongest value comes from combining topic prediction with entity and tone analysis.  
**Business interpretation:** This turns NewsBot into a practical intelligence workflow for monitoring, triage, and insight generation.


In [ ]:
insights = {
    "dataset_overview": {
        "articles": int(len(df)),
        "categories": SELECTED_CATEGORIES,
        "category_distribution": {
            key: int(value)
            for key, value in df["category"].value_counts().reindex(SELECTED_CATEGORIES).items()
        },
        "average_word_count": float(df["word_count"].mean()),
        "average_character_count": float(df["char_count"].mean()),
    },
    "tfidf_insights": {
        category: [item["term"] for item in top_terms_by_category[category][:5]]
        for category in SELECTED_CATEGORIES
    },
    "pos_insights": {
        "highest_proper_noun_category": pos_category_summary["proper_nouns"].idxmax(),
        "highest_adjective_category": pos_category_summary["adjectives"].idxmax(),
    },
    "syntax_insights": {
        "longest_average_sentences": syntax_category_summary["avg_sentence_length"].idxmax(),
        "most_noun_phrases": syntax_category_summary["noun_phrase_count"].idxmax(),
    },
    "sentiment_insights": {
        "most_positive_category": sentiment_summary["mean_sentiment"].idxmax(),
        "most_negative_category": sentiment_summary["mean_sentiment"].idxmin(),
        "average_sentiment_by_category": {
            key: float(value)
            for key, value in sentiment_summary["mean_sentiment"].items()
        },
    },
    "classification_performance": {
        "best_model": best_model_name,
        "deployment_model": deployment_model_name,
        "model_metrics": model_results_df.to_dict(orient="records"),
    },
    "entity_insights": {
        "entity_type_counts": {
            key: int(value) for key, value in entities_df["label"].value_counts().items()
        },
        "most_common_entities": top_entities.to_dict(orient="records"),
    },
    "business_recommendations": [
        "Use the classifier as a first-pass routing layer for editors or analysts.",
        "Track sentiment shifts within each category to monitor coverage tone over time.",
        "Use entity extraction to build searchable dashboards around people, organizations, and places.",
        "Pair TF-IDF and entity trends to identify emerging subtopics within major categories.",
    ],
    "limitations": [
        "The dataset uses headlines plus short descriptions, not full article bodies.",
        "Sentiment is lexicon-based, so it may miss subtle sarcasm or domain-specific tone.",
        "The final model is a strong baseline but not a transformer-based state-of-the-art system.",
    ],
}

with open(OUTPUT_DIR / "comprehensive_insights.json", "w", encoding="utf-8") as handle:
    json.dump(insights, handle, indent=2)

print("Integrated summary")
print("-" * 60)
print(f"Dataset used: HuffPost News Category Dataset balanced sample ({len(df)} articles)")
print(f"Best model by Macro F1: {best_model_name}")
print(f"Deployment model: {deployment_model_name}")
print(
    f"Most positive category: {insights['sentiment_insights']['most_positive_category']} | "
    f"Most negative category: {insights['sentiment_insights']['most_negative_category']}"
)
print(
    f"Highest proper noun usage: {insights['pos_insights']['highest_proper_noun_category']} | "
    f"Longest sentences: {insights['syntax_insights']['longest_average_sentences']}"
)
print("Top TF-IDF terms by category:")
for category, terms in insights["tfidf_insights"].items():
    print(f"- {category}: {', '.join(terms)}")


## Final System Integration

**What was done:** Wrapped the trained classifier, preprocessing, sentiment, and entity extraction into a reusable NewsBot class.  
**Why it matters:** A final integrated object shows that the notebook is a coherent NLP system rather than a sequence of unrelated experiments.  
**Key finding:** New articles can now move through the same pipeline used during training and evaluation.  
**Business interpretation:** This is the closest section to a deployable prototype because it accepts fresh text and returns actionable outputs.


In [ ]:
class NewsBotIntelligenceSystem:
    def __init__(self, classifier, vectorizer, nlp_model, feature_mode="tfidf", numeric_scaler=None):
        self.classifier = classifier
        self.vectorizer = vectorizer
        self.nlp = nlp_model
        self.feature_mode = feature_mode
        self.numeric_scaler = numeric_scaler

    def preprocess_article(self, title, content):
        full_text = f"{title} {content}".strip()
        processed_text = preprocess_text(full_text)
        return full_text, processed_text

    def build_numeric_features(self, title, full_text):
        sentiment_scores = analyze_sentiment(full_text)
        return np.array(
            [
                [
                    sentiment_scores["compound"],
                    sentiment_scores["pos"],
                    sentiment_scores["neu"],
                    sentiment_scores["neg"],
                    len(full_text),
                    len(full_text.split()),
                    len(title),
                ]
            ]
        )

    def classify_article(self, title, content):
        full_text, processed_text = self.preprocess_article(title, content)
        tfidf_features = self.vectorizer.transform([processed_text])
        if self.feature_mode == "combined":
            numeric_features = self.build_numeric_features(title, full_text)
            if self.numeric_scaler is not None:
                numeric_features = self.numeric_scaler.transform(numeric_features)
            model_features = hstack([tfidf_features, csr_matrix(numeric_features)], format="csr")
        else:
            model_features = tfidf_features

        predicted_category = self.classifier.predict(model_features)[0]
        probabilities = self.classifier.predict_proba(model_features)[0]
        category_probabilities = dict(zip(self.classifier.classes_, probabilities))
        confidence = float(np.max(probabilities))

        return predicted_category, confidence, category_probabilities, full_text

    def extract_entities(self, text):
        doc = self.nlp(text)
        entities = []
        for ent in doc.ents:
            if ent.label_ not in TARGET_ENTITY_LABELS:
                continue
            entities.append(
                {
                    "text": ent.text,
                    "label": ent.label_,
                    "description": spacy.explain(ent.label_),
                }
            )
        return entities

    def analyze_article_sentiment(self, text):
        return analyze_sentiment(text)

    def generate_insights(self, category, confidence, entities, sentiment):
        insights = []

        if confidence >= 0.80:
            insights.append(f"High-confidence routing candidate for {category}.")
        else:
            insights.append("Borderline routing result. Manual review would be appropriate.")

        if sentiment["compound"] >= 0.05:
            insights.append("Tone is overall positive.")
        elif sentiment["compound"] <= -0.05:
            insights.append("Tone is overall negative.")
        else:
            insights.append("Tone is mostly neutral.")

        if entities:
            dominant_labels = Counter(entity["label"] for entity in entities).most_common(2)
            insights.append(
                "Main entity focus: " + ", ".join(f"{label} ({count})" for label, count in dominant_labels)
            )
        else:
            insights.append("No major named entities were extracted from the text.")

        return insights

    def process_article(self, title, content):
        category, confidence, category_probabilities, full_text = self.classify_article(title, content)
        entities = self.extract_entities(full_text)
        sentiment = self.analyze_article_sentiment(full_text)
        insights = self.generate_insights(category, confidence, entities, sentiment)

        return {
            "title": title,
            "predicted_category": category,
            "category_confidence": confidence,
            "category_probabilities": category_probabilities,
            "sentiment": sentiment,
            "entities": entities,
            "statistics": {
                "word_count": len(full_text.split()),
                "character_count": len(full_text),
                "entity_count": len(entities),
            },
            "insights": insights,
        }

newsbot = NewsBotIntelligenceSystem(
    classifier=deployment_model,
    vectorizer=tfidf_vectorizer,
    nlp_model=nlp,
    feature_mode=deployment_feature_mode,
    numeric_scaler=numeric_scaler,
)

print("Integrated NewsBot system is ready.")


In [ ]:
test_articles = [
    {
        "title": "Senators Debate New Federal Climate Spending Package",
        "content": "Lawmakers in Washington argued over a climate and infrastructure proposal after administration officials said the plan would reshape energy investment over the next decade.",
    },
    {
        "title": "Startup Raises $180 Million To Expand Cybersecurity Platform",
        "content": "Investors backed the software company after strong enterprise demand, and executives said the new capital would support hiring, product expansion, and international growth.",
    },
    {
        "title": "Star Forward Scores Late Winner In Championship Match",
        "content": "The packed stadium erupted after the final-minute goal secured the title, while the coach praised the squad's discipline and defensive shape throughout the tournament.",
    },
    {
        "title": "Streaming Platform Announces Sci-Fi Series From Award-Winning Director",
        "content": "The company unveiled a new original production slate and said the headline series would debut globally next spring with a cast of well-known film and television actors.",
    },
    {
        "title": "Researchers Unveil Smaller AI Model Built For Mobile Devices",
        "content": "Engineers said the system delivers fast on-device language processing with lower energy use, opening the door for new productivity and accessibility features.",
    },
    {
        "title": "Doctors Recommend Consistent Sleep Routines During Stressful Periods",
        "content": "Health specialists said regular sleep schedules, lighter evening screen use, and basic exercise can reduce burnout symptoms and improve daily recovery.",
    },
]

print("Testing the final NewsBot system")
print("-" * 60)

for article in test_articles:
    result = newsbot.process_article(article["title"], article["content"])
    top_probabilities = sorted(
        result["category_probabilities"].items(),
        key=lambda item: item[1],
        reverse=True,
    )[:3]

    print(f"Title: {result['title']}")
    print(f"Predicted category: {result['predicted_category']} ({result['category_confidence']:.2%})")
    print(
        "Top probabilities: "
        + ", ".join(f"{label}={score:.3f}" for label, score in top_probabilities)
    )
    print(
        f"Sentiment: {result['sentiment']['sentiment_label']} "
        f"(compound={result['sentiment']['compound']:.3f})"
    )

    if result["entities"]:
        entity_preview = ", ".join(
            f"{entity['text']} [{entity['label']}]"
            for entity in result["entities"][:5]
        )
    else:
        entity_preview = "No major named entities"

    print(f"Entities: {entity_preview}")
    print(f"Statistics: {result['statistics']}")
    print("Insights:")
    for insight in result["insights"]:
        print(f"- {insight}")
    print("-" * 60)


## Bonus Features

In addition to the required notebook deliverable, this project also includes a bonus interactive extension designed to support extra-credit opportunities.

### Interactive Dashboard Bonus

A Streamlit app was built as a companion interface for this project. The app uses the same dataset sample, model outputs, and NewsBot logic so the project can be demonstrated in a more interactive format instead of only as a static notebook.

The bonus app includes:

- a project dashboard summarizing the final dataset, model comparison, and key findings
- a live NewsBot article analyzer where a user can enter a new headline and description
- a dataset explorer for browsing the sampled news articles
- a gallery of saved notebook visualizations

### Advanced Analysis Bonus

The Streamlit app also adds a temporal-trends section that extends the original notebook analysis. This view compares article volume and average sentiment by year and category, which provides a stronger business-intelligence angle and supports the advanced-analysis bonus opportunity.

### Hosting and Demonstration Value

Because the Streamlit app is deployment-ready, the project can be hosted online through Streamlit Community Cloud after the repository is pushed to GitHub. That makes the project easier to demonstrate in a portfolio or class submission and shows that the notebook analysis can be presented as a lightweight interactive application.


## Project Summary and Next Steps

This notebook meets the core assignment requirements by integrating:

- real-world problem framing
- preprocessing and normalization
- TF-IDF feature analysis
- POS and syntax analysis
- sentiment analysis
- multi-class classification
- named entity recognition
- a final end-to-end NewsBot pipeline

### Final Summary

- **Dataset:** HuffPost News Category Dataset using a balanced 1,800-article sample across six categories
- **Article text used:** headline + short description
- **Modeling outcome:** three classifiers were compared; Linear SVM produced the best Macro F1 score, while Multinomial Naive Bayes was retained for the interactive NewsBot because it also provides class probabilities and stayed close in performance
- **Linguistic insight:** category differences show up in both vocabulary and structure
- **Sentiment insight:** tone varies across categories and can support editorial monitoring
- **Entity insight:** named entities add searchable context beyond simple labels

### Limitations

- The source data is shorter than full news articles.
- Sentiment is lexicon-based rather than transformer-based.
- The balanced sample is designed for classroom efficiency, not full-corpus production scale.

### Future Improvements

- test transformer-based classifiers for higher accuracy
- compare additional categories or add temporal trend analysis
- build a lightweight Streamlit or Gradio interface as a bonus extension
- expand entity relationship analysis into a graph or dashboard
